# Семинар 9. Метод Ньютона: вычислительные задачи

Ноутбук основан на лекции 9 и сопоставлен с бумажным листком задач **один к одному**: программная задача 1 развивает теоретическую задачу 1 и так далее. Задачи 1–5 — основные, 6–10 — дополнительные.

**Работа:** заполните пометки `TODO`, выполните код и после каждого эксперимента напишите 2–3 предложения с выводом. Для основной части достаточно NumPy и Matplotlib; данные создаются внутри ноутбука, скачивание не требуется. Не используйте явное обращение матрицы для вычисления шага Ньютона: решайте линейную систему.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=6, suppress=True)
rng = np.random.default_rng(9)
plt.rcParams.update({"figure.figsize": (7, 4), "axes.grid": True, "grid.alpha": 0.2})
print("NumPy:", np.__version__)

## Основные задачи

### Задача 1. Квадратичная модель нелинейной функции

Аналог теоретической задачи 1. Для $f(x,y)=x^2/2+x^4/4+2y^2$ и $z_0=(1,1)$:

а) реализуйте градиент и гессиан аналитически; б) найдите направление Ньютона через `np.linalg.solve`; в) на графике по $\alpha$ сравните истинную функцию $f(z_0+\alpha p)$ и квадратичную модель $m_{z_0}(\alpha p)$; г) объясните, почему эти кривые касаются при $\alpha=0$, но не совпадают всюду.

In [ ]:
z01 = np.array([1.0, 1.0])

def f1(z):
    x, y = z
    return 0.5*x*x + 0.25*x**4 + 2*y*y

def grad1(z):
    # TODO: вернуть массив формы (2,)
    return None

def hess1(z):
    # TODO: вернуть матрицу формы (2, 2)
    return None

if grad1(z01) is not None and hess1(z01) is not None:
    g1, H1 = grad1(z01), hess1(z01)
    p1 = np.linalg.solve(H1, -g1)
    alpha1 = np.linspace(0, 1.5, 150)
    # TODO: вычислить массивы true1 и model1 по alpha1; model1 использует g1, H1
    true1, model1 = None, None
    if true1 is not None and model1 is not None:
        plt.plot(alpha1, true1, label="исходная функция")
        plt.plot(alpha1, model1, "--", label="квадратичная модель")
        plt.axvline(1, color="gray", linestyle=":", label="полный шаг")
        plt.xlabel(r"$\alpha$"); plt.ylabel("значение"); plt.legend(); plt.show()
        print("направление:", p1, "разница при полном шаге:", true1[np.argmin(abs(alpha1-1))]-model1[np.argmin(abs(alpha1-1))])

### Задача 2. Один шаг на квадратичной функции

Аналог теоретической задачи 2. Возьмите те же $A$, $b$ и $z_0$. Реализуйте точный шаг Ньютона и проверьте его из **20 случайных начальных точек**. Затем сравните одну траекторию с градиентным спуском при $\alpha=1/\lambda_{\max}(A)$. Постройте график нормы ошибки — это наглядно отделяет свойство задачи от удачного отдельного старта.

In [ ]:
A2 = np.array([[4.0, 1.0], [1.0, 2.0]])
b2 = np.array([1.0, 3.0])
z02 = np.array([2.0, -1.0])
zstar2 = np.linalg.solve(A2, b2)

def newton_step2(z):
    # TODO: вычислить градиент и решить систему A2 @ p = -g
    return None

starts2 = rng.normal(size=(20, 2))
if newton_step2(z02) is not None:
    reached2 = np.array([newton_step2(z) for z in starts2])
    print("максимальная ошибка одного шага:", np.max(np.linalg.norm(reached2-zstar2, axis=1)))
    assert np.allclose(reached2, zstar2)
    # TODO: выполнить 25 шагов градиентного спуска из z02 и построить ошибку по итерациям

### Задача 3. Когда направление Ньютона убывает

Аналог теоретической задачи 3. Для 100 случайных положительно определённых матриц $H$ и ненулевых $g$ вычислите $p$ из $Hp=-g$, сохраните значения $g^\top p$ и проверьте их знак. Затем повторите расчёт для $q(x,y)=x^2-y^2/2+x+y$ в нуле и сравните $q(p)$ с $q(0)$. Сформулируйте, какое предположение меняется.

In [ ]:
def q3(z):
    x, y = z
    return x*x - 0.5*y*y + x + y

H3_bad = np.diag([2.0, -1.0])
g3_bad = np.array([1.0, 1.0])

# TODO: в цикле сгенерировать H = B.T @ B + 0.1*I и случайный g; записать g @ p
slopes3_good = None
# TODO: решить систему для H3_bad, g3_bad; найти наклон и изменение q3
p3_bad = None
if slopes3_good is not None and p3_bad is not None:
    print("максимальный наклон при H>0:", np.max(slopes3_good))
    print("неопределённый H: наклон", g3_bad @ p3_bad,
          "изменение функции", q3(p3_bad)-q3(np.zeros(2)))
    assert np.max(slopes3_good) < 0

### Задача 4. Полный шаг и поиск Армихо

Аналог теоретической задачи 4. Для $f(x)=\sqrt{1+x^2}$, $x_0=2$, $c=0.1$, $\beta=0.5$ реализуйте шаг Ньютона и backtracking. Выведите таблицу испытанных $\alpha$, значений $f(x_0+\alpha p)$, порога Армихо и решения «принят/отклонён». Постройте обе кривые по $\alpha$ на $[0,1]$. Объясните, почему отрицательная производная по направлению в нуле не спасает полный шаг.

In [ ]:
def f4(x):
    return np.sqrt(1+x*x)

def grad4(x):
    return x / np.sqrt(1+x*x)

def hess4(x):
    return 1 / (1+x*x)**1.5

x04, c4, beta4 = 2.0, 0.1, 0.5

def armijo4(x, p, c=0.1, beta=0.5):
    # TODO: пробовать alpha=1,beta,beta**2,...; вернуть (принятая alpha, журнал испытаний)
    return None, []

# TODO: вычислить p4 из f'(x04), f''(x04)
p4 = None
if p4 is not None:
    alpha4, trials4 = armijo4(x04, p4, c4, beta4)
    for row in trials4:
        print(row)
    if alpha4 is not None:
        aa = np.linspace(0, 1, 300)
        plt.plot(aa, f4(x04+aa*p4), label=r"$f(x_0+\alpha p)$")
        plt.plot(aa, f4(x04)+c4*aa*grad4(x04)*p4, "--", label="порог Армихо")
        plt.axvline(alpha4, color="green", linestyle=":", label="принятый шаг")
        plt.xlabel(r"$\alpha$"); plt.legend(); plt.show()

### Задача 5. Квадратичная скорость вблизи минимума

Аналог теоретической задачи 5. Для $f(x)=x^2/2+x^3/3$ реализуйте полный метод Ньютона из $x_0=1/4$. Выведите $x_k$, ошибку $e_k=|x_k|$ и отношения $e_{k+1}/e_k^2$, пока не начнётся потеря точности. Проверьте численно оценку $e_{k+1}\le 2e_k^2$ при $|x_k|\le 1/4$. Почему она является локальной?

In [ ]:
def grad5(x):
    return x + x*x

def hess5(x):
    return 1 + 2*x

def newton_map5(x):
    # TODO: вернуть x - grad5(x)/hess5(x)
    return None

if newton_map5(0.25) is not None:
    xs5 = [0.25]
    for _ in range(5):
        xs5.append(newton_map5(xs5[-1]))
    # TODO: напечатать таблицу ошибок и отношений; не делить на нулевую ошибку
    # TODO: сравнить все ненулевые ошибки с оценкой из листка

## Дополнительные задачи

### Задача 6. Вырожденный гессиан

Аналог теоретической задачи 6. Запустите метод Ньютона для $f(x)=x^4$ из нескольких $x_0\ne0$. На полулогарифмическом графике сравните $|x_k|$ с $(2/3)^k|x_0|$. Добавьте защиту от старта в $x=0$ и объясните, почему запись $0/0$ не означает необходимость делать шаг.

In [ ]:
def newton_step6(x, tol=1e-14):
    # TODO: если градиент достаточно мал, вернуть x; иначе решить скалярное уравнение Ньютона
    return None

starts6 = [1.0, -2.0, 0.1]
if newton_step6(1.0) is not None:
    # TODO: построить для каждого старта траекторию из 15 шагов и теоретическую прямую
    pass

### Задача 7. Проверка локальной оценки на сетке

В теоретической задаче 7 оценка $e_{k+1}\le (L_H/2m)e_k^2$ доказывается для общего случая. Проверьте её численно для $f(x)=e^x-x-1$ на $[-1/2,1/2]$, где можно взять $m=e^{-1/2}$, $L_H=e^{1/2}$. Постройте графики левой и правой частей. Исследуйте, насколько оценка точна по сравнению с фактической ошибкой.

In [ ]:
grid7 = np.linspace(-0.5, 0.5, 501)
m7, LH7 = np.exp(-0.5), np.exp(0.5)

def newton_map7(x):
    # TODO: вернуть полный шаг для f(x)=exp(x)-x-1
    return None

if newton_map7(0.25) is not None:
    e_next7 = np.abs(newton_map7(grid7))
    bound7 = LH7/(2*m7)*grid7**2
    assert np.all(e_next7 <= bound7 + 1e-12)
    # TODO: построить обе кривые и описать запас оценки

### Задача 8. Когда принимается полный шаг

Аналог теоретической задачи 8. Для $f(x)=e^x-x-1$ возьмите положительные старты $x\in\{0.5,0.2,0.05,0.01\}$ и сравните решение Армихо при $c=0.1$ и $c=0.6$, $\beta=0.5$. Затем повторите для квадратичной $q(x)=x^2/2$. Объясните, почему порог $c<1/2$ важен для гарантированного принятия полного шага вблизи минимума.

In [ ]:
starts8 = [0.5, 0.2, 0.05, 0.01]

def f8(x):
    return np.exp(x) - x - 1

def grad8(x):
    return np.exp(x) - 1

def hess8(x):
    return np.exp(x)

def backtracking8(f, grad, x, p, c, beta=0.5):
    # TODO: вернуть первую alpha из 1,beta,beta**2,..., удовлетворяющую Армихо
    return None

if backtracking8(f8, grad8, 0.5, -grad8(0.5)/hess8(0.5), 0.1) is not None:
    # TODO: вывести таблицу принятых alpha для обеих функций и обоих c
    pass

### Задача 9. Логистическая регрессия

Аналог теоретической задачи 9. Сначала проверьте аналитический градиент и гессиан на маленьком наборе из листка при $w=0$. Затем создайте случайную двумерную выборку и реализуйте демпфированный метод Ньютона с Армихо. На каждой итерации проверяйте минимальное собственное значение гессиана и сравните число итераций с градиентным спуском. Для устойчивого вычисления потери используйте `np.logaddexp`.

In [ ]:
X9_small = np.array([[1.0, 0.0], [1.0, 1.0]])
y9_small = np.array([1.0, -1.0])
lam9 = 1.0

def logistic9(w, X, y, lam):
    margins = y * (X @ w)
    return np.sum(np.logaddexp(0.0, -margins)) + 0.5*lam*np.dot(w, w)

def grad9(w, X, y, lam):
    # TODO: вернуть аналитический градиент
    return None

def hess9(w, X, y, lam):
    # TODO: вернуть аналитический гессиан
    return None

if grad9(np.zeros(2), X9_small, y9_small, lam9) is not None and hess9(np.zeros(2), X9_small, y9_small, lam9) is not None:
    g9 = grad9(np.zeros(2), X9_small, y9_small, lam9)
    H9 = hess9(np.zeros(2), X9_small, y9_small, lam9)
    print("g(0):", g9, "H(0):", H9, "p:", np.linalg.solve(H9, -g9))
    # TODO: создать выборку из 200 точек и сравнить траектории Newton+Armijo и GD

### Задача 10. Вырожденный МНК и Ridge

Аналог теоретической задачи 10. Возьмите матрицу из листка, найдите минимизатор через `np.linalg.lstsq`, проверьте ранг и поведение `np.linalg.solve(X.T @ X, X.T @ y)`. Добавьте Ridge-регуляризацию и сравните решения. Затем создайте матрицу с двумя почти совпадающими столбцами и сравните числа обусловленности $X$ и $X^\top X$; объясните практическую пользу QR/SVD.

In [ ]:
X10 = np.array([[1.0, 1.0], [0.0, 0.0]])
y10 = np.array([1.0, 0.0])
lam10 = 1.0

# TODO: получить решение минимальной нормы через np.linalg.lstsq
w10_min_norm = None
# TODO: решить Ridge-систему (X.T@X+lam*I)w=X.T@y
w10_ridge = None

if w10_min_norm is not None and w10_ridge is not None:
    print("rank:", np.linalg.matrix_rank(X10))
    print("минимальная норма:", w10_min_norm, "Ridge:", w10_ridge)
    # TODO: создать почти зависимые столбцы и сравнить cond(X), cond(X.T@X)

---

## Что сдавать

Для каждой выполненной задачи сохраните код, численный результат или график и короткий комментарий: **какое утверждение лекции подтверждено, при каких предпосылках и чего эксперимент не доказывает**. Дополнительные задачи можно выполнять после основных.